<a href="https://colab.research.google.com/github/mena-04/DoS-Stress-Testing/blob/main/testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!nvidia-smi
!pip uninstall -y torch torchvision torchaudio vllm
!pip install -q -U uv
!uv pip install --system vllm --torch-backend=cu130
import torch
import vllm

print("vLLM:", vllm.__version__)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Fri Sep 11 12:22:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

ImportError: libcudart.so.13: cannot open shared object file: No such file or directory

In [1]:
import torch
import vllm

print("vLLM:", vllm.__version__)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

vLLM: 0.29.0
Torch: 2.13.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: Tesla T4


In [2]:
import importlib.util

print("torchaudio installed:",
      importlib.util.find_spec("torchaudio") is not None)

torchaudio installed: True


In [3]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130


In [4]:
import subprocess, time, requests, os, signal

# Stop old server if it exists
try:
    server.terminate()
    server.wait(timeout=10)
except:
    pass

log = open("/content/vllm.log", "w")

server = subprocess.Popen(
    [
        "vllm", "serve",
        "Qwen/Qwen2.5-0.5B-Instruct",
        "--dtype", "half",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.85",
        "--max-num-seqs", "8",
        "--host", "127.0.0.1",
        "--port", "8000",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
)

print("PID:", server.pid)

PID: 6832


In [5]:
for i in range(90):
    if server.poll() is not None:
        print("SERVER PROCESS EXITED")
        break

    try:
        r = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )
        if r.status_code == 200:
            print("vLLM READY")
            break
    except:
        pass

    time.sleep(2)

vLLM READY


In [ ]:
!tail -n 100 /content/vllm.log

(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347] 
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.29.0
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:347] 
(APIServer pid=9154) INFO 09-11 10:53:03 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'host': '127.0.0.1', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85}
(APIServer pid=9154) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=9154) INFO 09-11 10:53:19 [model.py:684] Resolved 

In [6]:
import requests
import time

URL = "http://127.0.0.1:8000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Explain AI inference in one sentence."
        }
    ],
    "max_tokens": 32,
    "temperature": 0
}

start = time.perf_counter()

r = requests.post(
    URL,
    json=payload,
    timeout=60
)

elapsed = time.perf_counter() - start

print("status:", r.status_code)
print("latency:", round(elapsed, 3), "seconds")

data = r.json()

print("response:")
print(data["choices"][0]["message"]["content"])

print("usage:")
print(data["usage"])

status: 200
latency: 0.99 seconds
response:
AI inference involves processing and analyzing large amounts of data to make predictions or decisions based on patterns and relationships within that data.
usage:
{'prompt_tokens': 37, 'total_tokens': 62, 'completion_tokens': 25, 'prompt_tokens_details': None, 'completion_tokens_details': None}


In [7]:
m = requests.get(
    "http://127.0.0.1:8000/metrics",
    timeout=10
)

print("metrics status:", m.status_code)

wanted = [
    "vllm:num_requests_running",
    "vllm:num_requests_waiting",
    "vllm:request_success_total",
    "vllm:e2e_request_latency_seconds",
    "vllm:request_queue_time_seconds",
    "vllm:prompt_tokens_total",
    "vllm:generation_tokens_total",
]

for metric in wanted:
    print("\n---", metric, "---")
    for line in m.text.splitlines():
        if line.startswith(metric):
            print(line)

metrics status: 200

--- vllm:num_requests_running ---
vllm:num_requests_running{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0

--- vllm:num_requests_waiting ---
vllm:num_requests_waiting{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct",reason="capacity"} 0.0
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-0.5B-Instruct",reason="deferred"} 0.0

--- vllm:request_success_total ---
vllm:request_success_total{engine="0",finished_reason="stop",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 1.0
vllm:request_success_total{engine="0",finished_reason="length",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:request_success_total{engine="0",finished_reason="abort",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:request_success_total{engine="0",finished_reason="error",model_name="Qwen/Qwen2.5-0.5B-Instruct"} 0.0
vllm:request_success_total{engine="0",finished_reason="repetit

# background metrics sampler

In [8]:
import threading
import requests
import time
import re
import csv

stop_sampling = False
samples = []

def get_value(text, name):
    pattern = rf'^{re.escape(name)}\{{.*?\}}\s+([0-9.eE+-]+)'
    m = re.search(pattern, text, re.MULTILINE)
    return float(m.group(1)) if m else None

def sampler():
    while not stop_sampling:
        try:
            text = requests.get(
                "http://127.0.0.1:8000/metrics",
                timeout=2
            ).text

            samples.append({
                "timestamp": time.time(),
                "running": get_value(
                    text,
                    "vllm:num_requests_running"
                ),
                "waiting": get_value(
                    text,
                    "vllm:num_requests_waiting"
                )
            })
        except Exception:
            pass

        time.sleep(0.25)

thread = threading.Thread(target=sampler, daemon=True)
thread.start()

print("sampler started")

sampler started


# concurrent overload test

In [9]:
import asyncio
import aiohttp
import time
import statistics

URL = "http://127.0.0.1:8000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30

async def send_one(session, i):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "user", "content": EXPENSIVE_PROMPT}
        ],
        "max_tokens": 256,
        "temperature": 0
    }

    start = time.perf_counter()

    try:
        async with session.post(URL, json=payload, timeout=120) as r:
            await r.text()
            latency = time.perf_counter() - start
            return {
                "id": i,
                "status": r.status,
                "latency": latency
            }
    except Exception as e:
        return {
            "id": i,
            "status": "error",
            "latency": time.perf_counter() - start
        }



async def run_load(n=50):
    async with aiohttp.ClientSession() as session:
        tasks = [
            asyncio.create_task(send_one(session, i))
            for i in range(n)
        ]
        return await asyncio.gather(*tasks)

results = await run_load(40)

latencies = [
    r["latency"]
    for r in results
    if r["status"] == 200
]

print("completed:", len(results))
print("successful:", len(latencies))

if latencies:
    print("p50:", round(statistics.median(latencies), 3))

    sorted_lat = sorted(latencies)
    p95_index = int(0.95 * len(sorted_lat)) - 1
    print("p95:", round(sorted_lat[p95_index], 3))

    print("max:", round(max(latencies), 3))

completed: 40
successful: 40
p50: 7.677
p95: 12.177
max: 12.178


In [10]:
stop_sampling = True
thread.join(timeout=2)

print("samples:", len(samples))
print("max running:", max(x["running"] or 0 for x in samples))
print("max waiting:", max(x["waiting"] or 0 for x in samples))

samples: 148
max running: 8.0
max waiting: 32.0


# Load generator

In [11]:
!pip install -q locust
!locust --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.4/115.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.6/270.6 kB 25.3 MB/s eta 0:00:00
locust 2.46.5 from /usr/local/lib/python3.13/dist-packages/locust (Python 3.13.15)


In [12]:
%%writefile /content/locustfile.py
# first testing with normal traffic
from locust import HttpUser, task, between
import random


MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

SMALL_PROMPT = "Explain AI inference briefly."

MEDIUM_PROMPT = (
    "Explain how an AI inference server handles requests, batching, "
    "token generation, and resource usage. "
) * 10

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30


class LegitimateUser(HttpUser):
    wait_time = between(0.5, 1.5)
    weight = 4

    @task
    def legitimate_request(self):
        payload = {
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": SMALL_PROMPT
                }
            ],
            "max_tokens": 32,
            "temperature": 0
        }

        self.client.post(
            "/v1/chat/completions",
            json=payload,
            name="legitimate"
        )


class AttackerUser(HttpUser):
    wait_time = between(0.05, 0.15)
    weight = 1

    @task
    def attack_request(self):
        payload = {
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": EXPENSIVE_PROMPT
                }
            ],
            "max_tokens": 256,
            "temperature": 0
        }

        self.client.post(
            "/v1/chat/completions",
            json=payload,
            name="attacker"
        )

Writing /content/locustfile.py


In [13]:
!mkdir -p /content/results

In [16]:
!locust \
  -f /content/locustfile.py \
  --headless \
  --host http://127.0.0.1:8000 \
  --users 4 \
  --spawn-rate 2 \
  --run-time 15s \
  --csv /content/results/normal \
  --csv-full-history \
  LegitimateUser

[2026-09-11 12:38:48,166] 9d728fde6ca4/INFO/locust.main: Starting Locust 2.46.5
[2026-09-11 12:38:48,167] 9d728fde6ca4/INFO/locust.main: Run time limit set to 15 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-09-11 12:38:48,168] 9d728fde6ca4/INFO/locust.runners: Ramping to 4 users at a rate of 2.00 per second
[2026-09-11 12:38:49,170] 9d728fde6ca4/INFO/locust.runners: All users spawned: {"LegitimateUser": 4} (4 total users)
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     legitimate       7     0(0.00%) |    220     206     252    210 |  

In [17]:
import pandas as pd

df = pd.read_csv("/content/results/normal_stats.csv")
display(df)

,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Min Response Time,Max Response Time,Average Content Size,Requests/s,...,66%,75%,80%,90%,95%,98%,99%,99.9%,99.99%,100%
0,POST,legitimate,48,0,210,213.737979,201.095848,335.048178,870.0,3.452947,...,210,210,220,220,250,340,340,340,340,340
1,NaN,Aggregated,48,0,210,213.737979,201.095848,335.048178,870.0,3.452947,...,210,210,220,220,250,340,340,340,340,340


In [18]:
%%writefile /content/locustfile.py
# sudden spike
from locust import HttpUser, task, between, LoadTestShape

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

SMALL_PROMPT = "Explain AI inference briefly."

MEDIUM_PROMPT = (
    "Explain how an AI inference server handles requests, batching, "
    "token generation, and resource usage. "
) * 10

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30


class LegitimateUser(HttpUser):
    wait_time = between(0.5, 1.5)
    weight = 1

    @task
    def legitimate_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": SMALL_PROMPT
                    }
                ],
                "max_tokens": 32,
                "temperature": 0
            },
            name="legitimate"
        )


class AttackerUser(HttpUser):
    wait_time = between(0.05, 0.15)
    weight = 1

    @task
    def attack_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": MEDIUM_PROMPT
                    }
                ],
                "max_tokens": 128,
                "temperature": 0
            },
            name="attacker"
        )


class SpikeShape(LoadTestShape):

    def tick(self):
        run_time = self.get_run_time()

        # 0-10s: legitimate traffic only
        if run_time < 10:
            return (
                4,
                4,
                [LegitimateUser]
            )

        # 10-20s: sudden attacker spike
        if run_time < 20:
            return (
                24,
                20,
                [LegitimateUser, AttackerUser]
            )

        # 20-30s: attack stops, return to normal
        if run_time < 30:
            return (
                4,
                20,
                [LegitimateUser]
            )

        return None

Overwriting /content/locustfile.py


In [19]:
!locust \
  -f /content/locustfile.py \
  --headless \
  --host http://127.0.0.1:8000 \
  --csv /content/results/spike \
  --csv-full-history

[2026-09-11 12:43:51,330] 9d728fde6ca4/INFO/locust.main: Starting Locust 2.46.5
[2026-09-11 12:43:51,330] 9d728fde6ca4/INFO/locust.runners: Shape test starting.
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-09-11 12:43:51,331] 9d728fde6ca4/INFO/locust.runners: Shape worker starting
[2026-09-11 12:43:51,332] 9d728fde6ca4/INFO/locust.runners: Shape test updating to 4 users at 4.00 spawn rate
[2026-09-11 12:43:51,332] 9d728fde6ca4/INFO/locust.runners: Ramping to 4 users at a rate of 4.00 per second
[2026-09-11 12:43:51,332] 9d728fde6ca4/INFO/locust.runners: All users spawned: {"AttackerUser": 0, "LegitimateUser": 4} (4 total users)
Type     Name  # reqs      # fails |    Avg     

In [20]:
import pandas as pd

df = pd.read_csv("/content/results/spike_stats.csv")
display(df)

,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Min Response Time,Max Response Time,Average Content Size,Requests/s,...,66%,75%,80%,90%,95%,98%,99%,99.9%,99.99%,100%
0,POST,attacker,62,0,1400,1407.184731,992.891218,1945.796912,1413.709677,2.070404,...,1500,1500,1600,1600,1700,1900,1900,1900,1900,1900
1,POST,legitimate,137,0,570,587.842140,198.191600,1592.788929,870.000000,4.574925,...,750,830,880,1000,1400,1500,1500,1600,1600,1600
2,NaN,Aggregated,199,0,830,843.114706,198.191600,1945.796912,1039.396985,6.645330,...,1100,1300,1400,1500,1600,1700,1900,1900,1900,1900


In [21]:
hist = pd.read_csv("/content/results/spike_stats_history.csv")

display(hist.tail(20))

,Timestamp,User Count,Type,Name,Requests/s,Failures/s,50%,66%,75%,80%,...,99.9%,99.99%,100%,Total Request Count,Total Failure Count,Total Median Response Time,Total Average Response Time,Total Min Response Time,Total Max Response Time,Total Average Content Size
60,1789130655,4,POST,legitimate,7.5,0.0,750.0,830.0,860.0,900.0,...,1100.0,1100.0,1100.0,117,0,650,648.004295,198.191600,1592.788929,870.000000
61,1789130655,4,NaN,Aggregated,13.7,0.0,930.0,1300.0,1400.0,1500.0,...,1800.0,1800.0,1800.0,179,0,880,910.960647,198.191600,1945.796912,1058.324022
62,1789130656,4,POST,attacker,4.7,0.0,1500.0,1500.0,1500.0,1600.0,...,1700.0,1700.0,1700.0,62,0,1400,1407.184731,992.891218,1945.796912,1413.709677
63,1789130656,4,POST,legitimate,6.7,0.0,710.0,760.0,830.0,850.0,...,1000.0,1000.0,1000.0,121,0,630,636.863708,198.191600,1592.788929,870.000000
64,1789130656,4,NaN,Aggregated,11.4,0.0,850.0,1300.0,1400.0,1500.0,...,1700.0,1700.0,1700.0,183,0,850,897.846787,198.191600,1945.796912,1054.207650
65,1789130657,4,POST,attacker,4.2,0.0,1500.0,1500.0,1500.0,1600.0,...,1600.0,1600.0,1600.0,62,0,1400,1407.184731,992.891218,1945.796912,1413.709677
66,1789130657,4,POST,legitimate,6.1,0.0,650.0,760.0,780.0,850.0,...,1000.0,1000.0,1000.0,123,0,630,630.038595,198.191600,1592.788929,870.000000
67,1789130657,4,NaN,Aggregated,10.3,0.0,780.0,930.0,1400.0,1400.0,...,1600.0,1600.0,1600.0,185,0,850,890.487570,198.191600,1945.796912,1052.216216
68,1789130658,4,POST,attacker,4.2,0.0,1500.0,1500.0,1500.0,1500.0,...,1600.0,1600.0,1600.0,62,0,1400,1407.184731,992.891218,1945.796912,1413.709677
69,1789130658,4,POST,legitimate,6.1,0.0,570.0,690.0,760.0,760.0,...,930.0,930.0,930.0,125,0,620,624.403010,198.191600,1592.788929,870.000000


In [22]:
%%writefile /content/locustfile.py
# sustained flood
from locust import HttpUser, task, between, LoadTestShape

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

SMALL_PROMPT = "Explain AI inference briefly."

MEDIUM_PROMPT = (
    "Explain how an AI inference server handles requests, batching, "
    "token generation, and resource usage. "
) * 10

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30


class LegitimateUser(HttpUser):
    wait_time = between(0.5, 1.5)
    weight = 1

    @task
    def legitimate_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": SMALL_PROMPT
                    }
                ],
                "max_tokens": 32,
                "temperature": 0
            },
            name="legitimate"
        )


class AttackerUser(HttpUser):
    wait_time = between(0.05, 0.15)
    weight = 1

    @task
    def attack_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": MEDIUM_PROMPT
                    }
                ],
                "max_tokens": 128,
                "temperature": 0
            },
            name="attacker"
        )


class FloodShape(LoadTestShape):

    def tick(self):
        run_time = self.get_run_time()

        # 0-10s: normal legitimate traffic
        if run_time < 10:
            return (
                4,
                4,
                [LegitimateUser]
            )

        # 10-40s: sustained flood
        if run_time < 40:
            return (
                32,
                20,
                [LegitimateUser, AttackerUser]
            )

        # 40-50s: recovery
        if run_time < 50:
            return (
                4,
                20,
                [LegitimateUser]
            )

        return None

Overwriting /content/locustfile.py


In [23]:
!locust \
  -f /content/locustfile.py \
  --headless \
  --host http://127.0.0.1:8000 \
  --csv /content/results/flood \
  --csv-full-history

[2026-09-11 12:49:31,789] 9d728fde6ca4/INFO/locust.main: Starting Locust 2.46.5
[2026-09-11 12:49:31,790] 9d728fde6ca4/INFO/locust.runners: Shape test starting.
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-09-11 12:49:31,791] 9d728fde6ca4/INFO/locust.runners: Shape worker starting
[2026-09-11 12:49:31,791] 9d728fde6ca4/INFO/locust.runners: Shape test updating to 4 users at 4.00 spawn rate
[2026-09-11 12:49:31,791] 9d728fde6ca4/INFO/locust.runners: Ramping to 4 users at a rate of 4.00 per second
[2026-09-11 12:49:31,792] 9d728fde6ca4/INFO/locust.runners: All users spawned: {"AttackerUser": 0, "LegitimateUser": 4} (4 total users)
Type     Name  # reqs      # fails |    Avg     

In [24]:
import pandas as pd

df = pd.read_csv("/content/results/flood_stats.csv")
display(df)

hist = pd.read_csv("/content/results/flood_stats_history.csv")
display(hist.tail(30))

,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Min Response Time,Max Response Time,Average Content Size,Requests/s,...,66%,75%,80%,90%,95%,98%,99%,99.9%,99.99%,100%
0,POST,attacker,189,0,2100,2066.455697,955.254715,3210.393183,1414.00000,3.801247,...,2100,2200,2200,2300,2600,3000,3100,3200,3200,3200
1,POST,legitimate,298,0,1300,1136.721806,196.579744,2186.271675,870.00000,5.993500,...,1400,1500,1500,1700,1800,2000,2000,2200,2200,2200
2,NaN,Aggregated,487,0,1500,1497.542557,196.579744,3210.393183,1081.12115,9.794747,...,2000,2000,2100,2200,2300,2600,2800,3200,3200,3200


,Timestamp,User Count,Type,Name,Requests/s,Failures/s,50%,66%,75%,80%,...,99.9%,99.99%,100%,Total Request Count,Total Failure Count,Total Median Response Time,Total Average Response Time,Total Min Response Time,Total Max Response Time,Total Average Content Size
111,1789131012,12,POST,attacker,6.4,0.0,2100.0,2100.0,2200.0,2200.0,...,2400.0,2400.0,2400.0,189,0,2100,2066.455697,955.254715,3210.393183,1414.000000
112,1789131012,12,POST,legitimate,8.0,0.0,1400.0,1400.0,1400.0,1500.0,...,1600.0,1600.0,1600.0,268,0,1400,1241.211519,197.265116,2186.271675,870.000000
113,1789131012,12,NaN,Aggregated,14.4,0.0,1500.0,2000.0,2100.0,2100.0,...,2400.0,2400.0,2400.0,457,0,1600,1582.505063,197.265116,3210.393183,1094.980306
114,1789131013,4,POST,attacker,6.6,0.0,2100.0,2100.0,2200.0,2200.0,...,2400.0,2400.0,2400.0,189,0,2100,2066.455697,955.254715,3210.393183,1414.000000
115,1789131013,4,POST,legitimate,7.7,0.0,1400.0,1400.0,1500.0,1500.0,...,1600.0,1600.0,1600.0,270,0,1400,1233.530908,197.265116,2186.271675,870.000000
116,1789131013,4,NaN,Aggregated,14.3,0.0,1500.0,2000.0,2100.0,2100.0,...,2400.0,2400.0,2400.0,459,0,1600,1576.499939,197.265116,3210.393183,1094.000000
117,1789131014,4,POST,attacker,6.2,0.0,2100.0,2100.0,2200.0,2200.0,...,2400.0,2400.0,2400.0,189,0,2100,2066.455697,955.254715,3210.393183,1414.000000
118,1789131014,4,POST,legitimate,7.6,0.0,1400.0,1400.0,1400.0,1500.0,...,1600.0,1600.0,1600.0,274,0,1400,1218.573281,197.265116,2186.271675,870.000000
119,1789131014,4,NaN,Aggregated,13.8,0.0,1500.0,2000.0,2100.0,2100.0,...,2400.0,2400.0,2400.0,463,0,1500,1564.685109,197.265116,3210.393183,1092.064795
120,1789131015,4,POST,attacker,6.0,0.0,2100.0,2100.0,2200.0,2200.0,...,2400.0,2400.0,2400.0,189,0,2100,2066.455697,955.254715,3210.393183,1414.000000


In [26]:
%%writefile /content/locustfile.py
# low-and-slow
from locust import HttpUser, task, between, LoadTestShape

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

SMALL_PROMPT = "Explain AI inference briefly."

MEDIUM_PROMPT = (
    "Explain how an AI inference server handles requests, batching, "
    "token generation, and resource usage. "
) * 10

EXPENSIVE_PROMPT = (
    "Explain in detail the architecture of an AI inference server, "
    "including request scheduling, KV cache, GPU execution, batching, "
    "prefill, decode, memory usage, and latency. "
) * 30


class LegitimateUser(HttpUser):
    wait_time = between(0.5, 1.5)
    weight = 1

    @task
    def legitimate_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": SMALL_PROMPT
                    }
                ],
                "max_tokens": 32,
                "temperature": 0
            },
            name="legitimate"
        )


class AttackerUser(HttpUser):
    wait_time = between(1.5, 2.5)

    @task
    def attack_request(self):
        self.client.post(
            "/v1/chat/completions",
            json={
                "model": MODEL,
                "messages": [
                    {
                        "role": "user",
                        "content": EXPENSIVE_PROMPT
                    }
                ],
                "max_tokens": 256,
                "temperature": 0
            },
            name="attacker"
        )


class LowSlowShape(LoadTestShape):

    def tick(self):
        run_time = self.get_run_time()

        # 0-10s: normal traffic only
        if run_time < 10:
            return (
                4,
                4,
                [LegitimateUser]
            )

        # 10-40s: low-rate expensive attackers appear
        if run_time < 40:
            return (
                8,
                2,
                [LegitimateUser, AttackerUser]
            )

        # 40-50s: attacker disappears
        if run_time < 50:
            return (
                4,
                4,
                [LegitimateUser]
            )

        return None

Overwriting /content/locustfile.py


In [27]:
!locust \
  -f /content/locustfile.py \
  --headless \
  --host http://127.0.0.1:8000 \
  --csv /content/results/low_slow \
  --csv-full-history

[2026-09-11 13:02:07,486] 9d728fde6ca4/INFO/locust.main: Starting Locust 2.46.5
[2026-09-11 13:02:07,487] 9d728fde6ca4/INFO/locust.runners: Shape test starting.
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-09-11 13:02:07,488] 9d728fde6ca4/INFO/locust.runners: Shape worker starting
[2026-09-11 13:02:07,488] 9d728fde6ca4/INFO/locust.runners: Shape test updating to 4 users at 4.00 spawn rate
[2026-09-11 13:02:07,489] 9d728fde6ca4/INFO/locust.runners: Ramping to 4 users at a rate of 4.00 per second
[2026-09-11 13:02:07,489] 9d728fde6ca4/INFO/locust.runners: All users spawned: {"AttackerUser": 0, "LegitimateUser": 4} (4 total users)
Type     Name  # reqs      # fails |    Avg     

In [28]:
import pandas as pd

df = pd.read_csv("/content/results/low_slow_stats.csv")
display(df)

hist = pd.read_csv("/content/results/low_slow_stats_history.csv")
display(hist.tail(30))

,Type,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Min Response Time,Max Response Time,Average Content Size,Requests/s,...,66%,75%,80%,90%,95%,98%,99%,99.9%,99.99%,100%
0,POST,attacker,14,0,1800,1967.481917,1761.399410,3374.858091,2114.00000,0.279836,...,1800,1800,1900,2700,3400,3400,3400,3400,3400,3400
1,POST,legitimate,208,0,210,248.690992,196.388284,1146.079165,870.00000,4.157562,...,230,240,250,270,330,710,1000,1100,1100,1100
2,NaN,Aggregated,222,0,220,357.083212,196.388284,3374.858091,948.45045,4.437398,...,240,250,250,490,1800,1800,1900,3400,3400,3400


,Timestamp,User Count,Type,Name,Requests/s,Failures/s,50%,66%,75%,80%,...,99.9%,99.99%,100%,Total Request Count,Total Failure Count,Total Median Response Time,Total Average Response Time,Total Min Response Time,Total Max Response Time,Total Average Content Size
108,1789131768,4,POST,attacker,0.6,0.0,1800.0,1800.0,1800.0,1800.0,...,1800.0,1800.0,1800.0,14,0,1800,1967.481917,1761.399410,3374.858091,2114.000000
109,1789131768,4,POST,legitimate,4.7,0.0,230.0,240.0,240.0,250.0,...,300.0,300.0,300.0,176,0,220,256.350853,196.388284,1146.079165,870.000000
110,1789131768,4,NaN,Aggregated,5.3,0.0,230.0,240.0,250.0,260.0,...,1800.0,1800.0,1800.0,190,0,230,382.434194,196.388284,3374.858091,961.663158
111,1789131769,4,POST,attacker,0.4,0.0,1800.0,1800.0,1800.0,1800.0,...,1800.0,1800.0,1800.0,14,0,1800,1967.481917,1761.399410,3374.858091,2114.000000
112,1789131769,4,POST,legitimate,4.8,0.0,220.0,230.0,240.0,250.0,...,300.0,300.0,300.0,179,0,220,255.586996,196.388284,1146.079165,870.000000
113,1789131769,4,NaN,Aggregated,5.2,0.0,230.0,240.0,250.0,260.0,...,1800.0,1800.0,1800.0,193,0,230,379.765902,196.388284,3374.858091,960.238342
114,1789131770,4,POST,attacker,0.4,0.0,1800.0,1800.0,1800.0,1800.0,...,1800.0,1800.0,1800.0,14,0,1800,1967.481917,1761.399410,3374.858091,2114.000000
115,1789131770,4,POST,legitimate,4.6,0.0,220.0,230.0,240.0,250.0,...,300.0,300.0,300.0,183,0,220,254.499289,196.388284,1146.079165,870.000000
116,1789131770,4,NaN,Aggregated,5.0,0.0,220.0,230.0,250.0,250.0,...,1800.0,1800.0,1800.0,197,0,230,376.234095,196.388284,3374.858091,958.406091
117,1789131771,4,POST,attacker,0.4,0.0,1800.0,1800.0,1800.0,1800.0,...,1800.0,1800.0,1800.0,14,0,1800,1967.481917,1761.399410,3374.858091,2114.000000
